# Этап 3. Интеграция с LLM
## Банковский консультант: RAG-система

1. Выбор модели и промпт-инжиниринг
2. RAG-цепочка с историей диалога
3. Оптимизация качества: Self-Query, Multi-Query, реранкинг, цитирование источников

## 0. Инициализация -- подключение к pgvector и LLM

In [ ]:
import os
import json
import time
from dotenv import load_dotenv

load_dotenv()

# Загружаем конфиг из Этапов 1-2
with open("rag_config.json", encoding="utf-8") as f:
    config = json.load(f)

CONNECTION_STRING = config["connection_string"]
COLLECTION_NAME   = config["collection_name"]

# OpenRouter совместим с OpenAI SDK
# Зарегистрируйтесь на openrouter.ai и получите бесплатный API ключ
# Ключ сохраните в .env: OPENROUTER_API_KEY=sk-or-...
OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY", "your-openrouter-api-key")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

print("Конфигурация загружена:")
print(f"  Коллекция       : {COLLECTION_NAME}")
print(f"  Лучший ретривер : {config.get('best_retriever', 'Similarity Search')}")
api_ok = OPENROUTER_API_KEY != 'your-openrouter-api-key'
print(f"  OpenRouter API  : {'OK задан' if api_ok else 'задайте OPENROUTER_API_KEY в .env'}")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import PGVector
from langchain_openai import ChatOpenAI

# Эмбеддинги
embeddings_model = HuggingFaceEmbeddings(
    model_name=config["embedding_model"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)

# Векторное хранилище
vectorstore = PGVector(
    collection_name=COLLECTION_NAME,
    connection_string=CONNECTION_STRING,
    embedding_function=embeddings_model,
)

# LLM через OpenRouter (совместим с OpenAI API)
# mixtral-8x7b-instruct -- бесплатная модель на OpenRouter
llm = ChatOpenAI(
    model="mistralai/mixtral-8x7b-instruct",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    temperature=0.1,    # низкая температура -> точные фактические ответы
    max_tokens=1024,
)

print("LLM и векторное хранилище инициализированы")
print("   LLM        : mixtral-8x7b-instruct (via OpenRouter)")
print("   Temperature: 0.1  |  Max tokens: 1024")

---
## 1. Промпт-инжиниринг

### Разработка системного промпта

Требования к промпту:
- Отвечать **строго на основе контекста** (без галлюцинаций)
- Профессиональный, но понятный тон
- **Цитировать источники** при ответе
- Корректно обрабатывать случаи отсутствия информации
- Не давать финансовых советов вне своей компетенции

In [ ]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM_PROMPT = (
    "Вы -- виртуальный консультант банка 'Альфа-Финанс'.\n"
    "Ваша задача -- давать точные, полезные ответы на вопросы клиентов о банковских продуктах.\n"
    "\n"
    "ПРАВИЛА:\n"
    "1. Отвечайте ТОЛЬКО на основе предоставленного контекста из документов банка.\n"
    "2. Если информация в контексте отсутствует -- прямо скажите об этом и предложите "
    "обратиться к специалисту.\n"
    "3. Цитируйте источник в конце ответа в формате: [Источник: название документа].\n"
    "4. Не давайте советов о том, какой продукт лучше выбрать -- только факты.\n"
    "5. Если вопрос содержит несколько частей -- ответьте на каждую отдельно.\n"
    "6. Приводите конкретные цифры (ставки, суммы, сроки) из документов.\n"
    "7. Используйте профессиональный, но понятный язык.\n"
    "\n"
    "КОНТЕКСТ ИЗ ДОКУМЕНТОВ БАНКА:\n"
    "{context}"
)

# Промпт для диалоговой RAG-цепочки
CHAT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

print("Системный промпт создан")
print("\nПредпросмотр:")
print(SYSTEM_PROMPT.replace("{context}", "<...контекст из БД...>"))

In [ ]:
from langchain.schema import HumanMessage, SystemMessage

# Тест: прямой вызов LLM без ретривера
test_context = (
    "Процентные ставки по потребительскому кредиту:\n"
    "Базовая ставка: от 9,9% годовых.\n"
    "Для зарплатных клиентов: скидка 1%.\n"
    "При страховании жизни: снижение на 0,5%.\n"
    "Максимальная ставка: 24,9% годовых."
)

test_messages = [
    SystemMessage(content=SYSTEM_PROMPT.replace("{context}", test_context)),
    HumanMessage(content="Какая ставка по кредиту если я зарплатный клиент?"),
]

print("Тест прямого вызова LLM...")
t0 = time.time()
response = llm.invoke(test_messages)
elapsed  = time.time() - t0

print(f"\nОтвет LLM ({elapsed:.1f}с):")
print("-" * 45)
print(response.content)

---
## 2. RAG-цепочка с историей диалога

Объединяем ретривер + LLM в единый пайплайн с поддержкой:
- Многоходовых диалогов (история сообщений)
- Форматирования контекста из документов
- Обработки случаев отсутствия релевантной информации

In [5]:
from langchain.schema import Document, AIMessage
from typing import List

# Базовый ретривер из Этапа 2 (лучший по MRR=0.825)
base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

def format_context(docs: List[Document]):
    """
    Форматирует найденные документы в контекст для промпта.
    Возвращает: (текст_контекста, список_источников)
    """
    if not docs:
        return "Информация по данному запросу не найдена в базе знаний банка.", []

    context_parts = []
    sources = []

    for i, doc in enumerate(docs, 1):
        title = doc.metadata.get("title", "Документ банка")
        context_parts.append(f"[{i}] {title}:\n{doc.page_content}")
        if title not in sources:
            sources.append(title)

    return "\n\n".join(context_parts), sources


def check_relevance(docs: List[Document], threshold: int = 1) -> bool:
    """Проверяет, найдена ли хоть какая-то релевантная информация."""
    return len(docs) >= threshold

In [6]:
class BankingRAGChain:
    """
    Диалоговая RAG-цепочка для банковского консультанта.
    Поддерживает историю диалога, цитирование источников,
    graceful degradation при отсутствии информации.
    """

    def __init__(self, retriever, llm, prompt, max_history: int = 6):
        self.retriever    = retriever
        self.llm          = llm
        self.prompt       = prompt
        self.max_history  = max_history
        self.chat_history: List = []
        self.last_sources: List[str] = []

    def ask(self, question: str, verbose: bool = False) -> str:
        """Задаёт вопрос системе и получает ответ."""
        # 1. Ретрив
        docs = self.retriever.invoke(question)

        if verbose:
            print(f"  Найдено документов: {len(docs)}")
            for d in docs:
                print(f"    - {d.metadata.get('title')} | {d.metadata.get('product_type')}")

        # 2. Контекст
        context, sources = format_context(docs)
        self.last_sources = sources

        # 3. Graceful degradation
        if not check_relevance(docs):
            fallback = (
                "К сожалению, в базе знаний банка не найдено информации по вашему запросу. "
                "Пожалуйста, обратитесь к специалисту банка по телефону 8-800-XXX-XX-XX "
                "или посетите ближайшее отделение."
            )
            self._update_history(question, fallback)
            return fallback

        # 4. Формируем сообщения с историей
        messages = self.prompt.format_messages(
            context=context,
            chat_history=self.chat_history[-self.max_history:],
            question=question,
        )

        # 5. LLM
        response = self.llm.invoke(messages)
        answer   = response.content

        # 6. Цитирование если не добавлено моделью
        if sources and "[Источник:" not in answer:
            answer += f"\n\n[Источник: {', '.join(sources)}]"

        # 7. Обновляем историю
        self._update_history(question, answer)
        return answer

    def _update_history(self, question: str, answer: str):
        self.chat_history.append(HumanMessage(content=question))
        self.chat_history.append(AIMessage(content=answer))
        if len(self.chat_history) > self.max_history * 2:
            self.chat_history = self.chat_history[-self.max_history * 2:]

    def reset(self):
        self.chat_history = []
        self.last_sources = []
        print("История диалога сброшена.")


rag_chain = BankingRAGChain(
    retriever=base_retriever,
    llm=llm,
    prompt=CHAT_PROMPT,
    max_history=6,
)

In [ ]:
# Тест одиночного вопроса
print("ТЕСТ RAG-ЦЕПОЧКИ: одиночный вопрос")

question = "Какая максимальная сумма ипотеки в Москве?"
print(f"\nВопрос: {question}")

t0 = time.time()
answer = rag_chain.ask(question, verbose=True)
print(f"\nОтвет ({time.time()-t0:.1f}с):")
print(answer)

In [ ]:
# Тест многоходового диалога
rag_chain.reset()

print("ТЕСТ ДИАЛОГА: многоходовая беседа")

dialogue = [
    "Расскажи про семейную ипотеку -- ставка и условия",
    "А какой минимальный первоначальный взнос по ней?",
    "Можно ли использовать материнский капитал?",
    "Сколько времени займёт оформление?",
]

for i, q in enumerate(dialogue, 1):
    print(f"\n[Ход {i}] Вопрос: {q}")
    t0  = time.time()
    ans = rag_chain.ask(q)
    print(f"Ответ ({time.time()-t0:.1f}с):\n{ans}")

In [ ]:
# Тест graceful degradation
rag_chain.reset()

print("ТЕСТ: вопрос вне компетенции системы")

out_of_scope = "Какой курс доллара к рублю сегодня?"
print(f"\nВопрос: {out_of_scope}")
answer = rag_chain.ask(out_of_scope, verbose=True)
print(f"\nОтвет:\n{answer}")

---
## 3. Оптимизация качества ответов

### 3.1 Multi-Query Retrieval

Генерируем несколько перефразировок вопроса, чтобы расширить покрытие ретрива.
Это особенно помогает когда клиент формулирует вопрос нестандартно.

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.prompts import PromptTemplate
import logging

MULTI_QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template=(
        "Вы помогаете улучшить поиск информации в документах банка.\n"
        "Сгенерируйте 3 РАЗНЫХ варианта формулировки следующего вопроса клиента.\n"
        "Каждый вариант -- на отдельной строке, без нумерации и пояснений.\n"
        "\n"
        "Исходный вопрос: {question}\n"
        "\n"
        "3 альтернативных формулировки:"
    )
)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    prompt=MULTI_QUERY_PROMPT,
)

# Включаем логирование для просмотра перефразировок
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

print("Multi-Query Retriever создан")

# Демонстрация
test_q = "условия ипотеки для молодых семей"
print(f"\nТест Multi-Query для: '{test_q}'")
print("(в логах ниже видны сгенерированные перефразировки)")
mq_docs = multi_query_retriever.invoke(test_q)
print(f"\nНайдено уникальных документов: {len(mq_docs)}")
for doc in mq_docs:
    print(f"  - [{doc.metadata.get('product_type')}] {doc.metadata.get('title')}")

### 3.2 Self-Query Retrieval

LLM автоматически извлекает фильтры из текста запроса
и применяет их к векторному поиску.

In [ ]:
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever

metadata_field_info = [
    AttributeInfo(
        name="product_type",
        description=(
            "Тип банковского продукта. Значения: "
            "'credit' (кредиты), 'mortgage' (ипотека), "
            "'deposit' (вклады), 'requirements' (требования к заёмщикам), "
            "'faq' (часто задаваемые вопросы)"
        ),
        type="string",
    ),
    AttributeInfo(
        name="title",
        description="Название документа банка",
        type="string",
    ),
]

document_content_description = (
    "Документы банка Альфа-Финанс о банковских продуктах: "
    "кредиты, ипотека, вклады, требования к заёмщикам, FAQ."
)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True,
    search_kwargs={"k": 4},
)

print("Self-Query Retriever создан")
print("Автоматически извлекает фильтры из запроса на естественном языке\n")

test_queries_sq = [
    "ставки по вкладам банка",
    "требования к заёмщику для ипотеки",
]

for q in test_queries_sq:
    print(f"Self-Query: '{q}'")
    try:
        sq_docs = self_query_retriever.invoke(q)
        print(f"  Найдено: {len(sq_docs)} документов")
        for doc in sq_docs:
            print(f"  [{doc.metadata.get('product_type')}] {doc.metadata.get('title')}")
    except Exception as e:
        print(f"  Ошибка: {e}")
    print()

### 3.3 Реранкинг результатов через косинусное сходство

In [ ]:
import numpy as np

class EmbeddingReranker:
    """
    Реранкинг документов: переранжирует кандидатов
    по семантическому сходству эмбеддингов с запросом.
    """

    def __init__(self, embeddings_model, top_n: int = 3):
        self.embeddings = embeddings_model
        self.top_n      = top_n

    def rerank(self, query: str, docs: List[Document], verbose: bool = True) -> List[Document]:
        if not docs:
            return docs

        query_emb   = np.array(self.embeddings.embed_query(f"query: {query}"))
        scored_docs = []

        for doc in docs:
            doc_emb = np.array(self.embeddings.embed_query(f"passage: {doc.page_content}"))
            score   = float(np.dot(query_emb, doc_emb))  # косинусное сходство (нормализованы)
            scored_docs.append((score, doc))

        scored_docs.sort(key=lambda x: x[0], reverse=True)

        if verbose:
            print(f"  Реранкинг {len(docs)} -> топ-{self.top_n}:")
            for rank, (score, doc) in enumerate(scored_docs[:self.top_n], 1):
                print(f"    [{rank}] score={score:.4f} | {doc.metadata.get('title')}")

        return [doc for _, doc in scored_docs[:self.top_n]]


reranker = EmbeddingReranker(embeddings_model, top_n=3)

# Демонстрация
query        = "какие документы нужны для получения ипотеки"
initial_docs = base_retriever.invoke(query)

print(f"Запрос: '{query}'")
print(f"\nДо реранкинга ({len(initial_docs)} документов):")
for i, doc in enumerate(initial_docs, 1):
    print(f"  [{i}] {doc.metadata.get('title')} | {doc.metadata.get('product_type')}")

print(f"\nПосле реранкинга:")
reranked_docs = reranker.rerank(query, initial_docs)

### 3.4 Проверка ответа на соответствие источнику (Answer Grounding)

In [ ]:
def check_answer_grounding(answer: str, context_docs: List[Document], llm) -> dict:
    """
    LLM-judge: проверяет, основан ли ответ на предоставленном контексте.
    Возвращает оценку groundedness от 0.0 до 1.0.
    """
    context_text = "\n".join([doc.page_content for doc in context_docs[:3]])

    grounding_prompt = (
        "Оцените, насколько данный ОТВЕТ соответствует предоставленному КОНТЕКСТУ.\n\n"
        f"КОНТЕКСТ:\n{context_text[:1500]}\n\n"
        f"ОТВЕТ:\n{answer[:500]}\n\n"
        "Оцените по шкале 0.0 - 1.0:\n"
        "1.0 = полностью подтверждается контекстом\n"
        "0.5 = частично подтверждается\n"
        "0.0 = противоречит или содержит информацию вне контекста\n\n"
        "Ответьте ТОЛЬКО в формате:\n"
        "ОЦЕНКА: <число от 0.0 до 1.0>\n"
        "ПОЯСНЕНИЕ: <одно предложение>"
    )

    try:
        response    = llm.invoke([HumanMessage(content=grounding_prompt)])
        text        = response.content.strip()
        score       = 0.5
        explanation = "Не удалось распарсить ответ"

        for line in text.split("\n"):
            if line.startswith("ОЦЕНКА:"):
                try:
                    score = float(line.split(":")[1].strip())
                except Exception:
                    pass
            elif line.startswith("ПОЯСНЕНИЕ:"):
                explanation = line.split(":", 1)[1].strip()

        return {"score": score, "explanation": explanation}

    except Exception as e:
        return {"score": -1.0, "explanation": f"Ошибка: {e}"}


# Тест
print("Тест проверки соответствия ответа источнику...")
test_question = "Какой минимальный первоначальный взнос по ипотеке?"
test_docs     = base_retriever.invoke(test_question)
rag_chain.reset()
test_answer   = rag_chain.ask(test_question)

print(f"\nВопрос : {test_question}")
print(f"Ответ  : {test_answer[:300]}...")

result = check_answer_grounding(test_answer, test_docs, llm)
print(f"\nGrounding score : {result['score']:.2f}")
print(f"Пояснение       : {result['explanation']}")

### 3.5 Полная оптимизированная RAG-цепочка

In [14]:
class OptimizedBankingRAGChain(BankingRAGChain):
    """
    Расширенная RAG-цепочка:
    - Multi-Query для расширения покрытия ретрива
    - Реранкинг результатов
    - Проверка grounding ответа
    - Дедупликация документов
    """

    def __init__(self, retriever, multi_query_retriever, reranker, llm, prompt, max_history=6):
        super().__init__(retriever, llm, prompt, max_history)
        self.mq_retriever     = multi_query_retriever
        self.reranker         = reranker
        self.grounding_scores = []

    def _get_relevant_docs(self, question: str) -> List[Document]:
        """Расширенный ретрив: Multi-Query + дедупликация + реранкинг."""
        base_docs = self.retriever.invoke(question)
        try:
            mq_docs  = self.mq_retriever.invoke(question)
            all_docs = base_docs + mq_docs
        except Exception:
            all_docs = base_docs

        # Дедупликация по содержимому
        seen, unique_docs = set(), []
        for doc in all_docs:
            key = doc.page_content[:100]
            if key not in seen:
                seen.add(key)
                unique_docs.append(doc)

        # Реранкинг если документов больше top_n
        if len(unique_docs) > 3:
            return self.reranker.rerank(question, unique_docs, verbose=False)
        return unique_docs

    def ask(self, question: str, verbose: bool = False, check_grounding: bool = False) -> str:
        docs             = self._get_relevant_docs(question)
        context, sources = format_context(docs)
        self.last_sources = sources

        if not check_relevance(docs):
            fallback = (
                "К сожалению, в базе знаний банка не найдено информации по вашему запросу. "
                "Пожалуйста, обратитесь к специалисту банка по телефону 8-800-XXX-XX-XX."
            )
            self._update_history(question, fallback)
            return fallback

        messages = self.prompt.format_messages(
            context=context,
            chat_history=self.chat_history[-self.max_history:],
            question=question,
        )
        response = self.llm.invoke(messages)
        answer   = response.content

        if sources and "[Источник:" not in answer:
            answer += f"\n\n[Источник: {', '.join(sources)}]"

        if check_grounding:
            gr = check_answer_grounding(answer, docs, self.llm)
            self.grounding_scores.append(gr["score"])
            if verbose:
                print(f"  Grounding score: {gr['score']:.2f} | {gr['explanation']}")

        self._update_history(question, answer)
        return answer


optimized_chain = OptimizedBankingRAGChain(
    retriever=base_retriever,
    multi_query_retriever=multi_query_retriever,
    reranker=reranker,
    llm=llm,
    prompt=CHAT_PROMPT,
    max_history=6,
)

In [ ]:
# Сравнение базовой и оптимизированной цепочек
rag_chain.reset()
optimized_chain.reset()

test_questions = [
    "Есть ли штраф за досрочное погашение кредита?",
    "Какие возрастные требования к заёмщику по ипотеке?",
    "Можно ли пополнять вклад 'Максимальный доход'?",
]

print("СРАВНЕНИЕ: базовая vs оптимизированная цепочка")

timing_results = []
for question in test_questions:
    print(f"\nВопрос: {question}")

    t0      = time.time()
    basic   = rag_chain.ask(question)
    t_basic = time.time() - t0

    t0    = time.time()
    opt   = optimized_chain.ask(question, check_grounding=True, verbose=True)
    t_opt = time.time() - t0

    timing_results.append({"question": question, "basic_time": t_basic, "opt_time": t_opt})

    print(f"  Базовая ({t_basic:.1f}с):")
    print(f"  {basic[:250]}...")
    print(f"  Оптимизированная ({t_opt:.1f}с):")
    print(f"  {opt[:250]}...")

print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ СРАВНЕНИЯ")
print(f"{'Вопрос':<45} {'Базовая':>10} {'Опт.':>10}")
for r in timing_results:
    print(f"{r['question'][:44]:<44} {r['basic_time']:>9.1f}c {r['opt_time']:>9.1f}c")

if optimized_chain.grounding_scores:
    avg_gr = sum(optimized_chain.grounding_scores) / len(optimized_chain.grounding_scores)
    print(f"\nСредний Grounding Score оптимизированной цепочки: {avg_gr:.2f}")

**Следующий шаг -- Этап 4: Анализ и оптимизация**